# PH-Aligned, Lifestyle-Augmented CVD Risk Model
### Project Overview, Literature Review & Sprint Plan

**Notebook 00 of 06** — this notebook contains no modeling code. It exists to document *why* this project exists,
what already exists in this space, what gap it fills, and how the remaining notebooks are structured.

| Notebook | Purpose |
|---|---|
| `00_project_overview.ipynb` | This file — motivation, literature review, framework |
| `01_data_processing_and_cleaning.ipynb` | Load raw data, inspect, handle nulls/duplicates/types |
| `02_feature_engineering.ipynb` | Composite lifestyle score, PH-aligned BMI bins, comorbidity count, encoding |
| `03_model_training.ipynb` | Train/val split, Logistic Regression / Random Forest / LightGBM baselines |
| `04_model_validation.ipynb` | ROC-AUC, PR-AUC, F1, cross-validation, feature importance |
| `05_ph_calibration.ipynb` | Recalibrate predicted probabilities using published PH prevalence statistics |
| `06_globorisk_benchmark_and_results.ipynb` | Benchmark vs. Globorisk-PH scorer, final write-up |


## 1. Motivation & Problem Statement

Cardiovascular disease (CVD) is not a niche health issue in the Philippines — it is the single leading cause of
death in the country, with heart and vascular disease outranking cancer, respiratory disease, and diabetes combined
as causes of mortality. A large share of the national Non-Communicable Disease (NCD) burden is attributable to a
narrow set of modifiable risk factors: smoking, physical inactivity, poor diet, and alcohol use. As the country has
urbanized rapidly over the past few decades, these lifestyle risk factors have become more prevalent, and NCDs have
overtaken infectious disease as the dominant driver of premature death.

To respond to this, the Philippine Department of Health adopted the **PhilPEN protocol** and the **Globorisk**
equation — a validated, country-calibrated cardiovascular risk score used by primary care and barangay health
workers to triage patients into risk tiers. An existing implementation of this — the
[`cvd-risk-scorer`](https://github.com/projcjdevs/cvd-risk-scorer) repository — packages this equation into a
usable API with PH-calibrated coefficients and multilingual clinical guidance.

**The gap:** Globorisk and similar equations are, by design, simple and clinically validated — they take a narrow,
fixed set of inputs (age, sex, systolic BP, smoking status, BMI or cholesterol, diabetes status). They do not, and
structurally cannot, incorporate finer-grained lifestyle information: physical activity level, diet quality,
alcohol consumption patterns, or general/mental health status. This project asks a specific, testable question about
that gap.


## 2. Research Questions & Hypothesis

- **RQ1:** Can a supervised model trained on a broader set of clinical *and* lifestyle indicators reproduce the
  general risk stratification behavior of an equation-based score like Globorisk?
- **RQ2:** Do lifestyle features (physical activity, diet, alcohol use, self-rated health) add predictive value
  *beyond* the standard clinical variables that equations like Globorisk already use?

**Hypothesis:** Lifestyle features will provide meaningful incremental signal on top of the classic clinical
variables, particularly for individuals who fall in ambiguous or borderline clinical ranges where the equation's
fixed thresholds are less informative.

This project is **not** attempting to replace or outperform Globorisk as a clinical tool — it is not clinically
validated and is not intended for medical use. It is a data-driven complement that asks what an ML approach,
trained on richer features, would surface that a fixed equation cannot.


## 3. The Existing Tool: `cvd-risk-scorer`

Reviewed directly from the repository README (accessed 2026):

- A FastAPI application that estimates CVD risk using the **Globorisk equation**, calibrated for the Philippine
  population, combined with **PhilPEN** clinical guidance tiers.
- Produces multilingual output (Filipino, Cebuano, Hiligaynon, Ilocano, English) intended for use by barangay
  health workers and clinicians.
- Two input modes: *office mode* (age, sex, systolic BP, smoking, BMI) and *lab mode* (adds total cholesterol and
  diabetes status for a more precise estimate).
- Validated only for ages 40–80, per Globorisk's original derivation bounds; outside that range the tool
  explicitly declines to score and instead gives generic guidance.
- **Important distinction for this project:** this tool is entirely **equation-based and deterministic** — it
  computes a closed-form formula with fixed, published coefficients. It does not train on patient data, and it has
  no feature engineering or learned parameters. This is precisely why a supervised-learning project in the same
  space is not redundant with it — the two approaches are fundamentally different (calculation vs. learning).


## 4. Literature & Data Landscape Review

A review of the surrounding literature and data sources that inform this project's design:

**On Filipino CVD/metabolic risk specifically:**
- A cross-sectional analysis of the 8th Philippine National Nutrition Survey (NNS, 2013) examined diabetes
  prevalence and cardiometabolic risk profiles among young (20–44) vs. older (45+) Filipino adults, finding
  meaningfully different risk profiles by age group and highlighting that waist-height ratio may outperform BMI as
  a predictor of cardiovascular risk in Filipino populations, and that PH-specific obesity cutoffs are lower than
  WHO's global defaults.
- A separate NNS-based study decomposed socioeconomic inequality in CVD risk among Filipino adults aged 40–74,
  using the Globorisk equation itself to estimate 10-year CVD risk and analyzing which risk factors (blood
  pressure, glucose, cholesterol, smoking) drive inequality across wealth groups.
- A third NNS-based study examined dietary patterns and NCD risk factors (diabetes, hypertension, dyslipidemia,
  obesity) among ~19,900 Filipino adults, underscoring how diet quality specifically links to NCD outcomes in this
  population.

**On methodology/frameworks:**
- **Globorisk** — the underlying cardiovascular risk equation, designed to be recalibrated per-country using
  national CVD incidence and risk factor prevalence data, rather than retrained on patient-level microdata.
- **PhilPEN Protocol (DOH)** — the Philippine Package of Essential NCD Interventions, defining clinical guidance
  tiers by risk level for primary care use.
- **WHO HEARTS Technical Package** — WHO's global framework for CVD risk management in primary care settings.

**On the PH open-data gap (why we can't just find a PH training set):**
- The **National Nutrition Survey**, which contains exactly the clinical + lifestyle variables this project needs,
  is restricted-access microdata obtainable only via formal request to FNRI — not a public download.
- The **WHO STEPS survey**, the standard cross-country tool for exactly this kind of NCD risk factor data
  (tobacco, alcohol, diet, activity, BP, glucose, cholesterol), has been conducted in dozens of countries with
  public microdata releases, but the Philippines has not conducted a general adult STEPS survey.
- The Philippines has conducted the **Global Adult Tobacco Survey (GATS)**, but it is tobacco-specific (no BP,
  cholesterol, or BMI data) and the **Global School-based Student Health Survey (GSHS)**, which covers school-age
  children, not adults.
- **Conclusion:** no public, row-level, adult clinical+lifestyle dataset exists for the Philippines. This is a
  documented limitation of this project, addressed via the calibration approach in Sprint 07, not by fabricating
  or improperly merging incompatible datasets.

**Training dataset used:**
- **CDC BRFSS Heart Disease Health Indicators Dataset** (derived from the 2015 Behavioral Risk Factor Surveillance
  System, a US telephone health survey). ~253,680 responses, 21 predictor features plus a binary
  `HeartDiseaseorAttack` target. Feature groups include clinical variables (HighBP, HighChol, BMI, Diabetes,
  Stroke), lifestyle variables (Smoker, PhysActivity, Fruits, Veggies, HvyAlcoholConsump), access/socioeconomic
  variables (AnyHealthcare, NoDocbcCost, Education, Income), self-reported health (GenHlth, MentHlth, PhysHlth,
  DiffWalk), and demographics (Sex, Age). Known limitations: it is a US population, self-reported (subject to
  recall/social-desirability bias), and heavily class-imbalanced (~9% positive).


## 5. PH Calibration Mechanics (implemented in full in Notebook 05)

Since no PH patient-level microdata exists, "PH alignment" happens in two places, neither of which involves
fabricating or improperly merging data:

**a) At the feature engineering stage (Notebook 02):** clinical thresholds are re-set to PH/Asian-Pacific-relevant
cutoffs where the literature supports it — most notably BMI categories, since the NNS-based literature above
indicates Filipino obesity risk cutoffs sit lower than the WHO global default (roughly ~23/~27 vs. 25/30).

**b) At the output stage (Notebook 05):** after training on BRFSS data, the model's predicted probabilities are
recalibrated so that its *average* predicted risk aligns with published Philippine population statistics (e.g.
national hypertension, diabetes, smoking, and obesity prevalence from DOH/NNS reports), rather than reflecting the
US sample's base rates. This is conceptually the same move Globorisk itself makes when "calibrated for the
Philippines" — it does not retrain on Filipino patient microdata either; it recalibrates the equation's baseline
using country-level incidence statistics. Candidate techniques to implement and compare:
  - **Prior/intercept correction** — shift the model's decision threshold or log-odds output to match a known
    target prevalence (a standard technique when training and deployment population base rates differ).
  - **Sample reweighting** — reweight training examples so that feature distributions (age, sex, smoking rate)
    more closely resemble published PH demographic proportions before/while fitting the model.

Both are implemented and compared via calibration curves (predicted probability vs. observed frequency) in
Notebook 05 — this is treated as an explicit, visualized before/after comparison, not a black-box adjustment.


## 6. Validation & Success Criteria

Because the target is heavily imbalanced (~9% positive class), accuracy alone would be misleading (a model that
always predicts "no CVD" would already score ~91% accuracy). Evaluation instead centers on:

- **ROC-AUC** and **Precision-Recall AUC** as primary ranking metrics
- **Precision, Recall (sensitivity), and F1** at a chosen operating threshold — recall is especially relevant here,
  since missing a true at-risk patient is costlier than a false alarm in a screening context
- **Stratified k-fold cross-validation** to ensure metrics are stable, not a lucky single split
- **Calibration curves**, before and after the PH-adjustment step (Section 5)
- **Feature importance comparison** — checking whether the classic Globorisk inputs (age, sex, BP, smoking,
  cholesterol/BMI) rank highly (sanity check that the model learned real signal) *and* whether the added lifestyle
  features (activity, diet, alcohol, self-rated health) show meaningful importance beyond that core set — this is
  the direct empirical answer to RQ2
- **Synthetic patient benchmark** — running the `cvd-risk-scorer` repo's own test cases (Pablo, Lorna, Ernesto)
  through the trained model and comparing predicted risk against the Globorisk-PH tier for the same profiles


## 7. Limitations (stated upfront)

- Training data is a US population survey; no public Filipino patient-level clinical+lifestyle dataset exists at
  this time (NNS is restricted-access; PH has not run a general adult STEPS survey).
- PH calibration relies on published aggregate statistics, not row-level Filipino data — it is a principled
  approximation, not a substitute for locally-collected training data.
- BRFSS is self-reported survey data, subject to recall bias and social-desirability bias (e.g. under-reporting
  smoking or alcohol use).
- This project is an educational/portfolio exercise. Outputs are not clinically validated and must not be used for
  actual medical decision-making — the same caveat the `cvd-risk-scorer` repo itself states for its intended
  users.


## 8. References

- Globorisk CVD Risk Score — https://www.globorisk.org/
- PhilPEN Protocol, Department of Health — https://doh.gov.ph/sites/default/files/publications/PhilPEN%20Protocol.pdf
- WHO HEARTS Technical Package — https://www.who.int/teams/noncommunicable-diseases/cardiovascular-diseases/management/tools/hearts
- `cvd-risk-scorer` repository — https://github.com/projcjdevs/cvd-risk-scorer
- WHO NCD Microdata Repository (STEPS surveys) — https://extranet.who.int/ncdsmicrodata/index.php/home
- CDC BRFSS Heart Disease Health Indicators Dataset (Kaggle) — https://www.kaggle.com/datasets/alexteboul/heart-disease-health-indicators-dataset
- Cardiometabolic risk profile of young adults with diabetes in the Philippines (8th NNS) — https://pmc.ncbi.nlm.nih.gov/articles/PMC8214347/
- Risk factor contributions to socioeconomic inequality in cardiovascular risk in the Philippines — https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10092926/
- Dietary pattern and nutrient intakes in association with NCD risk factors among Filipino adults — https://pmc.ncbi.nlm.nih.gov/articles/PMC7397579/
